In [1]:
!pip install evaluate
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install bert_score
!pip install --upgrade regex

from evaluate import load
import pandas as pd

df = pd.read_csv("/Users/paolacherneva/Desktop/WU/SBWL Data Science/Applications of Data Science/Taras_Edward_Paola_new/12319305_Cherneva_Paola/results/rag_comparison.csv")

bertscore = load("bertscore")

models = {
    "baseline": "baseline_answer",
    "fine_tuned": "tuned_answer",
    "rag": "rag_answer"
}

for name, col in models.items():
    result = bertscore.compute(
        predictions=df[col].astype(str),
        references=df["correct_answer"].astype(str),
        lang="en"
    )
    
    avg_f1 = sum(result["f1"]) / len(result["f1"])
    
    print(name, "BERTScore:", round(avg_f1, 4))

Looking in indexes: https://download.pytorch.org/whl/cpu
     |████████████████████████████████| 287 kB 3.7 MB/s eta 0:00:01
  Attempting uninstall: regex
    Found existing installation: regex 2021.4.4
    Uninstalling regex-2021.4.4:
      Successfully uninstalled regex-2021.4.4


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


baseline BERTScore: 0.8258
fine_tuned BERTScore: 0.8268
rag BERTScore: 0.7848


In [2]:
df[f"{name}_bertscore"] = result["f1"]

df.to_csv("results_with_bertscore.csv", index=False)

In [3]:
!pip install rouge-score

     |████████████████████████████████| 135 kB 6.5 MB/s eta 0:00:01
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24956 sha256=5ae352296e597d2f4af7d2bb9aaf6edb2917e445925cee705270af57830f5d2b
  Stored in directory: /Users/paolacherneva/Library/Caches/pip/wheels/24/55/6f/ebfc4cb176d1c9665da4e306e1705496206d08215c1acd9dde
Successfully built rouge-score


In [8]:
rouge = load("rouge")
results = []

for name, col in models.items():
    result = rouge.compute(
        predictions=df[col].astype(str),
        references=df["correct_answer"].astype(str),
    )
    score = round(result["rougeL"], 4)
    print(name, "ROUGE-L:", round(result["rougeL"], 4))
    results.append({"model": name, "ROUGE-L": score})

results_df = pd.DataFrame(results)
results_df.to_csv("rouge_results.csv", index=False)


baseline ROUGE-L: 0.1434
fine_tuned ROUGE-L: 0.1451
rag ROUGE-L: 0.0549
